In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
import sys
import astropy as ast
from astropy.io import ascii
from astropy.table import Table
import matplotlib.pyplot as plt
import pandas as pd
import glob
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.path as mpath
from pathlib import Path
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import AutoMinorLocator
fourpointstar = mpath.Path.unit_regular_star(4)
sevenpointstar = mpath.Path.unit_regular_star(7)
eightpointstar = mpath.Path.unit_regular_star(8)
from metrics import fit_plotter_flux, compute_lomb_scargle, plot_periodogram, info, plot_phase_fold_binned, plot_phase_fold

In [3]:
stats = pd.read_csv('summary_results03162026.csv')
df_smc = stats[stats['RA'] < 40]
df_lmc = stats[stats['RA'] > 40]
coords = pd.read_csv('merged_smc_lmc_coords_all.csv', comment='#', sep="\\s+", names=['RA', 'DEC'])
hrd_zoning = pd.read_csv('./HR_and_case-by-case/HRD_zoning_priority_03232026.csv')
# temps = pd.read_csv('../synth_phot_temp_estimation/ysg_temp_fitting_summary_v10_prefinal.csv')
temps = pd.read_csv('synth_phot_temp_estimation/ysg_temp_fitting_summary_03152026.csv')
var = pd.read_csv('summary_results03162026.csv')
hrd_priority = pd.read_csv("HR_and_case-by-case/HRD_zoning_priority_03232026.csv")
magellan_spectroscopy_orig = pd.read_csv("magellan_spectroscopy/magellan_spectroscopy_to_original_matches.csv")
magellan_spectroscopy_prefinal = pd.read_csv("magellan_spectroscopy/magellan_spectroscopy_to_prefinal_matches.csv")
plt.rcParams['font.family'] = 'serif'

In [4]:
print(((var['logT'] > 3.62) & (var['logL'] > 4.0) & (var['logT'] < 4.0) & (var['best_period'].notnull())).sum())
print(((temps['final_logT_mean'] > 3.62) & (temps['final_logL_mean'] > 4.0) & (temps['final_logT_mean'] < 4.0)).sum())

827
881


In [5]:
def mag_amplitude_plot(column, clim):
    plt.figure(figsize=(14, 6))
    c = stats[column]
    mask = (stats['offset_flag'] == 0).values
    variable_mask = (stats['alarm_level_flag'] == 0).values
    nonvariable_mask = (stats['alarm_level_flag'] == 1).values
    # Combine both masks: only select rows where both are True
    var_mask = mask & variable_mask
    nonvar_mask = mask & nonvariable_mask
    sc2 = plt.scatter(stats['overall_mean_mag'][nonvar_mask], stats['mag_amplitude_1'][nonvar_mask], c='gray', alpha=0.7, edgecolors='none', label='Non-periodic', zorder=1)
    sc1 = plt.scatter(stats['overall_mean_mag'][var_mask], stats['mag_amplitude_1'][var_mask], c=c[var_mask], cmap='inferno', alpha=0.6, edgecolors='black', label = 'Periodic Variable', zorder=2)
    # sc2 = plt.scatter(stats['overall_mean_mag'][nonvar_mask], stats['mag_amplitude_1'][nonvar_mask], c='gray', alpha=0.5, edgecolors='none', label='Non-variable')
    plt.clim(0, clim)
    cbar = plt.colorbar(sc1)
    cbar.set_label('Dominant Period [days]', fontsize=20)
    cbar.ax.tick_params(labelsize=20)
    plt.legend(fontsize=20)
    plt.tick_params(axis='both', which='both', direction='in', labelsize=20)
    # plt.xticks(fontsize=20)
    # plt.yticks(fontsize=20)
    plt.xlabel('Mean Magnitude (g)', fontsize=20)
    plt.ylabel('Amplitude [mag]', fontsize=20)
    plt.grid(True, zorder=0, alpha=0.1)
    # plt.savefig(f'hr_diagrams/mag_amp_period.pdf', dpi=300, bbox_inches='tight')
    plt.show()

In [6]:
def HR_highlight(index):
    fig = plt.figure(figsize=(8, 6))
    min= 0
    max = 0.4
    # box =  [307,286,756, 1189] #1168
# box3 = [716,541,579,1100]
    # norm = LogNorm(vmin=min, vmax=max)
    variable_mask = (var['alarm_level_flag']== 0)

    plt.scatter(var.loc[index, 'logT'], var.loc[index, 'logL'], c =var.loc[index, 'mag_amplitude_1'], cmap='magma_r', marker='*', s=100, edgecolors='cyan', label=f'Target {index}', vmin = min, vmax = max, zorder=10)
    plt.scatter(var['logT'][variable_mask], var['logL'][variable_mask], c =var['mag_amplitude_1'][variable_mask],  cmap='magma_r', marker='o', s=5, label='All targets', vmin = min, vmax = max)
    plt.colorbar(label='Amplitude (mags)')
    plt.xlim(3.55, 4.0)
    plt.ylim(4.0,5.5)
    plt.gca().invert_xaxis()
    plt.tick_params(axis='both', which='both', direction = 'in', labelsize=16)
    plt.xlabel('log(T) [K]')
    plt.ylabel('log(L) [Lsun]')
    plt.title(f'HRD with highlighted star {index}')
    plt.legend()
    plt.show()


In [7]:
chosen = [40, 156, 250, 307, 522, 613,  716,   780,  890, 953, 1085, 1130]
for i in chosen:
    # print coordinates
    print(f"{var.loc[i, 'RA']},{var.loc[i, 'DEC']}")
    # make dataframe of all coordinates
    coords_df = pd.DataFrame({'RA': var.loc[chosen, 'RA'], 'DEC': var.loc[chosen, 'DEC']})
print(coords_df)

11.78622,-73.236649
13.440647,-72.894165
14.487046,-73.562134
15.993419,-71.90551
76.149149,-68.748634
80.009071,-68.063393
83.291798,-67.733246
85.284794,-67.3564
13.979501,-72.675148
16.370598,-72.67263
78.390618,-66.571976
84.715095,-69.13533
             RA        DEC
40    11.786220 -73.236649
156   13.440647 -72.894165
250   14.487046 -73.562134
307   15.993419 -71.905510
522   76.149149 -68.748634
613   80.009071 -68.063393
716   83.291798 -67.733246
780   85.284794 -67.356400
890   13.979501 -72.675148
953   16.370598 -72.672630
1085  78.390618 -66.571976
1130  84.715095 -69.135330


In [9]:
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.gaia import Gaia

# Example
# chosen = pd.DataFrame({
#     "ra": [120.1234, 121.5678],
#     "dec": [-30.1234, -29.8765]
# })

results = []

for _, row in coords_df.iterrows():
    coord = SkyCoord(row["RA"], row["DEC"], unit="deg")

    job = Gaia.cone_search_async(coord, radius=1*u.arcsec)
    tbl = job.get_results()

    if len(tbl) > 0:
        # Closest source (results are ordered by distance)
        results.append(tbl[0])
        
    else:
        results.append(None)

/opt/anaconda3/envs/ysg/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]


In [12]:
print("Gaia DR3 results:")
# print(results)
for i, res in enumerate(results):
    if res is not None:
        # print(f"Source {i}: {res['source_id']}, Gmag: {res['phot_g_mean_mag']}")
        print(f"Source {i}: RA: {res['ra']}, DEC: {res['dec']}")
        print(f"Gaia ID: {res['source_id']}, Gmag: {res['phot_g_mean_mag']}")
        print(f"Candidate coords: {var.loc[chosen[i], 'RA']}, {var.loc[chosen[i], 'DEC']}")
        #print hms
        gaia_coord = SkyCoord(res['ra'], res['dec'], unit='deg')
        print(f"GAIA: RA (hms): {gaia_coord.ra.to_string(unit=u.hour, sep=':')}, "f"DEC (dms): {gaia_coord.dec.to_string(unit=u.deg, sep=':')}")
        print(f"Cand coords: RA (hms): {SkyCoord(var.loc[chosen[i], 'RA'], var.loc[chosen[i], 'DEC'], unit='deg').ra.to_string(unit=u.hour, sep=':')}, DEC (dms): {SkyCoord(var.loc[chosen[i], 'RA'], var.loc[chosen[i], 'DEC'], unit='deg').dec.to_string(unit=u.deg, sep=':')}")
        print("\n")
    else:
        print(f"Source {i}: No match found")

Gaia DR3 results:
Source 0: RA: 11.786262268058483, DEC: -73.2366454318955
Gaia ID: 4685850173680692480, Gmag: 12.65213394165039
Candidate coords: 11.78622, -73.236649
GAIA: RA (hms): 0:47:08.70294433, DEC (dms): -73:14:11.92355482
Cand coords: RA (hms): 0:47:08.6928, DEC (dms): -73:14:11.9364


Source 1: RA: 13.440708725802459, DEC: -72.89417396369485
Gaia ID: 4685955486272265984, Gmag: 12.5078706741333
Candidate coords: 13.440647, -72.894165
GAIA: RA (hms): 0:53:45.77009419, DEC (dms): -72:53:39.0262693
Cand coords: RA (hms): 0:53:45.75528, DEC (dms): -72:53:38.994


Source 2: RA: 14.487020565015289, DEC: -73.56215669620948
Gaia ID: 4685697380231799296, Gmag: 12.32644271850586
Candidate coords: 14.487046, -73.562134
GAIA: RA (hms): 0:57:56.8849356, DEC (dms): -73:33:43.76410635
Cand coords: RA (hms): 0:57:56.89104, DEC (dms): -73:33:43.6824


Source 3: RA: 15.99338709955404, DEC: -71.90552373191787
Gaia ID: 4690527667928185088, Gmag: 12.70322322845459
Candidate coords: 15.993419, -71

In [ ]:
mag_amplitude_plot('best_period', 100)

In [ ]:
index = 299
info(index)
HR_highlight(index)

# burgundy box:

In [ ]:
for index in [729, 849, 205, 299, 851, 584, 856, 914, 961, 882]:
    HR_highlight(index)

# bright stars without spectroscopy on the mag-amp diagram:

In [ ]:
for index in [1261, 1103, 946, 1253, 1115, 864, 965]:
    HR_highlight(index)
    info(index)

In [ ]:
info(951)
HR_highlight(951)

In [ ]:
# highL_cands_cutdown = [1020,1028,1029,1037,1055,1058,1087,1095,1102,1117,1120,1131,1187,1195,1201,1203,1211,1212,1213]
highL_cands = [403,534,585,596,602,725,741,744, 785,877,889, 1166, 1252, 1212, 1187,1037,1029]
for star in highL_cands:
    info(star)
    HR_highlight(star)